# Volumes and Polar Graphs (Demos)

This notebook explores **area between curves**, **volumes of revolution** (disk, washer, cylindrical shell), and **polar graphs** using interactive demos. **You can type your own functions** in the text boxes: use **x** for Cartesian (e.g. `x**2`, `sqrt(x)`, `sin(x)`) and **theta** for polar (e.g. `2*cos(theta)`, `1 + sin(theta)`).

**Learning objectives:**
1. **Area between curves** – set up and interpret $\int_a^b (f(x)-g(x))\,dx$.
2. **Disk and washer method** – volume by rotating a region about the x-axis; inner/outer radius for washers.
3. **Cylindrical shell method** – volume as $2\pi \int (\text{radius})(\text{height})\,dx$.
4. **Polar graphs** – plot $r = f(\theta)$ and area $\frac12\int r^2\,d\theta$.

---
## 1. Area between curves

**Key idea:** The area between $y = f(x)$ and $y = g(x)$ from $x = a$ to $x = b$ is $\int_a^b |f(x)-g(x)|\,dx$. If $f \geq g$ on $[a,b]$, this is $\int_a^b (f(x)-g(x))\,dx$.

**Use the demo below.** Type expressions for $f(x)$ and $g(x)$ (variable **x**), and set $a$, $b$. The shaded area and its numerical value are shown.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Text, Dropdown
from scipy.integrate import quad
from sympy import sympify, lambdify, Symbol
%matplotlib inline

def make_func(expr_str, var_name='x'):
    """Parse expression string and return (callable, None) or (None, error_message)."""
    if not expr_str or not expr_str.strip():
        return None, "Empty expression"
    try:
        var = Symbol(var_name)
        expr = sympify(expr_str.strip())
        func = lambdify([var], expr, modules=['numpy'])
        return func, None
    except Exception as e:
        return None, str(e)

def area_between_demo(f_expr='x**2', g_expr='x', a=0, b=1):
    func_f, err_f = make_func(f_expr, 'x')
    func_g, err_g = make_func(g_expr, 'x')
    a, b = min(a, b), max(a, b)
    
    if err_f:
        print(f"Could not parse f(x): {err_f}. Use variable x, e.g. x**2, sqrt(x), sin(x).")
        func_f = lambda x: x**2
    if err_g:
        print(f"Could not parse g(x): {err_g}. Use variable x.")
        func_g = lambda x: x
    
    x = np.linspace(a, b, 300)
    try:
        yf, yg = func_f(x), func_g(x)
    except Exception:
        print("Error evaluating functions on [a,b]. Try different a, b or expressions.")
        return
    
    area, _ = quad(lambda t: abs(func_f(t) - func_g(t)), a, b)
    
    plt.figure(figsize=(9, 5))
    plt.plot(x, yf, 'b-', lw=2, label='$f(x)$')
    plt.plot(x, yg, 'g-', lw=2, label='$g(x)$')
    plt.fill_between(x, yf, yg, alpha=0.3)
    plt.axvline(a, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(b, color='gray', linestyle='--', alpha=0.7)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.legend(loc='best', fontsize=9)
    plt.title(f'Area between curves = {area:.4f}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Area = {area:.6f}')

interact(area_between_demo,
         f_expr=Text(value='x**2', description='f(x)=', placeholder='x**2'),
         g_expr=Text(value='x', description='g(x)=', placeholder='x'),
         a=FloatSlider(value=0, min=-2, max=2, step=0.25, description='a'),
         b=FloatSlider(value=1, min=-1, max=3, step=0.25, description='b'));

**Question 1.** How does the area change when you swap $f$ and $g$? What must be true of $f$ and $g$ on $[a,b]$ for the formula $\int_a^b (f-g)\,dx$ to give the correct area?

*Use a **Markdown** cell below for your answer.*

*Your answer:*

---
## 2. Disk and washer method (volume)

**Key idea:** Rotating the region under $y = f(x)$ about the x-axis gives **disk** volume $V = \pi \int_a^b f(x)^2\,dx$. If the region is between $y = g(x)$ and $y = f(x)$ (with $f \geq g \geq 0$), the **washer** volume is $V = \pi \int_a^b (f(x)^2 - g(x)^2)\,dx$.

**Use the demo below.** Enter $f(x)$ (outer) and $g(x)$ (inner; use `0` for disk). Choose Disk or Washer and set $a$, $b$.

In [ ]:
def disk_washer_demo(f_expr='x**2', g_expr='0', method='Disk', a=0, b=1):
    func_f, err_f = make_func(f_expr, 'x')
    if not g_expr.strip() or g_expr.strip() == '0':
        func_g = lambda x: np.zeros_like(x)
        err_g = None
    else:
        func_g, err_g = make_func(g_expr, 'x')
    if err_f:
        print(f"Could not parse f(x): {err_f}. Use variable x.")
        func_f = lambda x: x**2
    if err_g:
        print(f"Could not parse g(x): {err_g}. Use 0 for disk.")
        func_g = lambda x: np.zeros_like(x)
    
    a, b = min(a, b), max(a, b)
    
    if method == 'Disk':
        vol, _ = quad(lambda t: np.pi * func_f(t)**2, a, b)
    else:
        vol, _ = quad(lambda t: np.pi * (func_f(t)**2 - func_g(t)**2), a, b)
    
    x = np.linspace(a, b, 300)
    yf = func_f(x)
    yg = func_g(x) if method == 'Washer' else np.zeros_like(x)
    
    plt.figure(figsize=(9, 5))
    plt.plot(x, yf, 'b-', lw=2, label='$f(x)$')
    if method == 'Washer':
        plt.plot(x, yg, 'g-', lw=2, label='$g(x)$')
        plt.fill_between(x, yg, yf, alpha=0.3)
    else:
        plt.fill_between(x, 0, yf, alpha=0.3)
    plt.axhline(0, color='k', lw=0.5)
    plt.axvline(a, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(b, color='gray', linestyle='--', alpha=0.7)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.legend(loc='best', fontsize=9)
    plt.title(f'{method} method: volume = {vol:.4f}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Volume = {vol:.6f}')

interact(disk_washer_demo,
         f_expr=Text(value='x**2', description='f(x)=', placeholder='x**2'),
         g_expr=Text(value='0', description='g(x)=', placeholder='0 for disk'),
         method=Dropdown(options=['Disk', 'Washer'], value='Disk', description='Method'),
         a=FloatSlider(value=0, min=0, max=2, step=0.25, description='a'),
         b=FloatSlider(value=1, min=0.5, max=3, step=0.25, description='b'));

**Question 2.** When is the washer method needed instead of the disk method? What do the outer radius $R(x)$ and inner radius $r(x)$ represent in the plot?

*Use a **Markdown** cell below for your answer.*

*Your answer:*

---
## 3. Cylindrical shell method

**Key idea:** For rotation about the **y-axis**, the volume of the solid formed by the region under $y = f(x)$ between $x = a$ and $x = b$ is $V = 2\pi \int_a^b x\, f(x)\,dx$ (radius $x$, height $f(x)$).

**Use the demo below.** Enter $f(x)$ and set $a$, $b$. The region and the volume are shown.

In [ ]:
def shell_demo(f_expr='x**2', a=0, b=1):
    func_f, err_f = make_func(f_expr, 'x')
    if err_f:
        print(f"Could not parse f(x): {err_f}. Use variable x.")
        func_f = lambda x: x**2
    
    a, b = min(a, b), max(a, b)
    vol, _ = quad(lambda t: 2 * np.pi * t * func_f(t), a, b)
    
    x = np.linspace(max(0.001, a), b, 300)
    yf = func_f(x)
    
    plt.figure(figsize=(9, 5))
    plt.plot(x, yf, 'b-', lw=2, label='$f(x)$')
    plt.fill_between(x, 0, yf, alpha=0.3)
    plt.axhline(0, color='k', lw=0.5)
    plt.axvline(a, color='gray', linestyle='--', alpha=0.7)
    plt.axvline(b, color='gray', linestyle='--', alpha=0.7)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.legend(loc='best', fontsize=9)
    plt.title(f'Cylindrical shell: V = 2 pi int x f(x) dx = {vol:.4f}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Volume = {vol:.6f}')

interact(shell_demo,
         f_expr=Text(value='x**2', description='f(x)=', placeholder='x**2'),
         a=FloatSlider(value=0, min=0, max=2, step=0.25, description='a'),
         b=FloatSlider(value=1, min=0.5, max=3, step=0.25, description='b'));

**Question 3.** How does the shell method differ from the disk method when rotating about the y-axis? Why does the integrand have a factor of $x$?

*Use a **Markdown** cell below for your answer.*

*Your answer:*

---
## 4. Polar graphs and area

**Key idea:** A polar curve $r = f(\theta)$ is plotted via $(x, y) = (r\cos\theta, r\sin\theta)$. The area enclosed from $\theta = \alpha$ to $\theta = \beta$ is $\frac12 \int_\alpha^\beta r^2\,d\theta$.

**Use the demo below.** Enter $r(\theta)$ using the variable **theta** (e.g. `2*cos(theta)`, `1 + sin(theta)`). Set the angle bounds $\alpha$ and $\beta$.

In [ ]:
def polar_demo(r_expr='2*cos(theta)', alpha=0, beta=3.14159):
    func_r, err = make_func(r_expr, 'theta')
    if err:
        print(f"Could not parse r(theta): {err}. Use variable theta, e.g. 2*cos(theta), 1+sin(theta).")
        func_r = lambda t: 2*np.cos(t)
    
    alpha, beta = min(alpha, beta), max(alpha, beta)
    th = np.linspace(alpha, beta, 400)
    try:
        r = func_r(th)
    except Exception:
        print("Error evaluating r(theta). Try different expression or bounds.")
        return
    
    area = 0.5 * quad(lambda t: func_r(t)**2, alpha, beta)[0]
    x = r * np.cos(th)
    y = r * np.sin(th)
    
    plt.figure(figsize=(7, 7))
    plt.plot(x, y, 'b-', lw=2, label='r(theta)')
    plt.fill(x, y, alpha=0.3)
    plt.axhline(0, color='k', lw=0.5)
    plt.axvline(0, color='k', lw=0.5)
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.legend(loc='best', fontsize=9)
    plt.title(f'Polar area = (1/2) int r^2 d(theta) = {area:.4f}')
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    print(f'Area = {area:.6f}')

interact(polar_demo,
         r_expr=Text(value='2*cos(theta)', description='r(theta)=', placeholder='2*cos(theta)'),
         alpha=FloatSlider(value=0, min=-3.15, max=3.15, step=0.2, description='alpha'),
         beta=FloatSlider(value=3.14159, min=-3.15, max=6.3, step=0.2, description='beta'));

**Question 4.** How does changing the angle interval $[\alpha, \beta]$ change the curve and the area? Give an example of a simple $r(\theta)$ and describe the shape.

*Use a **Markdown** cell below for your answer.*

*Your answer:*

---
## Summary

- **Area between curves:** $\int_a^b |f(x)-g(x)|\,dx$; use $\int_a^b (f-g)\,dx$ when $f \geq g$ on $[a,b]$.
- **Disk method:** $V = \pi \int_a^b f(x)^2\,dx$ (region under $y=f(x)$ rotated about x-axis).
- **Washer method:** $V = \pi \int_a^b (R(x)^2 - r(x)^2)\,dx$ with outer radius $R$ and inner radius $r$.
- **Cylindrical shell method:** $V = 2\pi \int_a^b x\,f(x)\,dx$ (rotation about y-axis).
- **Polar area:** $\frac12 \int_\alpha^\beta r^2\,d\theta$ for $r = f(\theta)$.

Remember: in the text boxes use **x** for Cartesian and **theta** for polar (e.g. `x**2`, `sqrt(x)`, `2*cos(theta)`).